In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# Gaussian Elimintaion

In [2]:
A = np.array([[0, 1, 5, -3, 3],
              [-2, 0, 0, 1, 3],
              [7, 0, 1, 0, -2],
              [0, 3, -1, 2, 4]])
A

array([[ 0,  1,  5, -3,  3],
       [-2,  0,  0,  1,  3],
       [ 7,  0,  1,  0, -2],
       [ 0,  3, -1,  2,  4]])

In [3]:
# rng = np.random.default_rng()
# a = rng.integers(low=-9, high=9, size=20).reshape(4, 5)
# a = np.zeros((4, 5)).astype(np.int8)
print(A)

def leading_zeros(A: np.ndarray) -> np.ndarray:
    '''
    Returns the array of number of leading zeros of an array, per row
    '''
    non_zero_mask = (A != 0)
    first_nonzero_index = np.argmax(non_zero_mask, axis=1)
    
    all_zeros = ~non_zero_mask.any(axis=1)
    
    counts = np.where(all_zeros, A.shape[1], first_nonzero_index)
    
    return counts

leading_zeros(A)

[[ 0  1  5 -3  3]
 [-2  0  0  1  3]
 [ 7  0  1  0 -2]
 [ 0  3 -1  2  4]]


array([1, 0, 0, 1])

In [21]:
A = np.array([[0, 1, 5, -3, 3],
              [-2, 0, 0, 1, 3],
              [7, 0, 1, 0, -2],
              [0, 3, -1, 2, 4]])

def exchange_rows(A: np.ndarray, i: int, j: int) -> np.ndarray:
    '''
    Exchange rows i and j of A.
    '''
    A[[i, j]] = A[[j, i]]
    return A

print(A)
exchange_rows(A, 0, 2)

[[ 0  1  5 -3  3]
 [-2  0  0  1  3]
 [ 7  0  1  0 -2]
 [ 0  3 -1  2  4]]


array([[ 7,  0,  1,  0, -2],
       [-2,  0,  0,  1,  3],
       [ 0,  1,  5, -3,  3],
       [ 0,  3, -1,  2,  4]])

In [111]:
A = np.array([[ 1, -2,  0,  0, -2],
              [ 0,  0,  1,  0,  1],
              [ 0,  0,  0,  0,  0],
              [-0, -0, -0,  1, -2]])

def is_ref(A: np.ndarray) -> bool:
    '''
    Checks if an array A is in its Row Echelon Form (REF) or not.
    
    For REF:
        - all rows that contain only zeros are at the bottom.
        - for non-zero rows, first non-zero element from left (pivot), is always right to the pivot of the row above.
    '''
    leading = leading_zeros(A)
    all_zero_rows = (leading == A.shape[1])

    if all_zero_rows.any():
        if np.any(np.diff(leading) < 0):
            return False
    
    return np.all(np.diff(leading_zeros(A[~all_zero_rows])) > 0)

def is_rref(A: np.ndarray) -> bool:
    '''
    Checks if an array A is in its Reduced Row Echelon Form (RREF) or not.

    For RREF:
        - it must be a REF.
        - every pivot must be 1.
        - the pivot must be the only non-zero entry in the column.
    '''
    if is_ref(A):
        A_filtered = A[~(leading_zeros(A) == A.shape[1])]
        non_zero_mask = (A_filtered != 0)
        first_non_zero_index = np.argmax(non_zero_mask, axis=1)
        return (A_filtered.T[first_non_zero_index] == np.eye(A_filtered.shape[0])).all()
    else:
        return False

# print(A)
is_ref(A)

False

In [114]:
A = np.array([[-2, 4, -2, -1, 4],
              [4, -8, 3, -3, 1],
              [1, -2, 1, -1, 1],
              [1, -2, 0, -3, 4]])

def gaussian_elimination(A: np.ndarray) -> np.ndarray:
    '''
    Converts any matrix A to its RREF using gaussian elimination.
    '''
    A_rref = A.copy().astype(np.float64)
    
    for i in range(4):
        A_rref = A_rref[np.argsort(leading_zeros(A_rref))]
        print(A_rref)
        
        non_zero_mask = (A_rref != 0)
        first_non_zero_index = np.argmax(non_zero_mask, axis=1)
    
        first_non_zero_values = A_rref[np.arange(A_rref.shape[0]), first_non_zero_index]

        # suggested by codex
        if first_non_zero_values[i] == 0:
            break
        
        A_rref[i] /= first_non_zero_values[i]

        pivot_col = first_non_zero_index[i] # suggested by codex
        need_zeroing = (A_rref[:, pivot_col] != 0)
        need_zeroing[i] = False
        
        if np.any(need_zeroing):
            A_rref[need_zeroing] -= (A_rref[need_zeroing, first_non_zero_index[i]][:, np.newaxis] * A_rref[i][np.newaxis, :])
        
        print(A_rref)
        print("–" * 20)

    return A_rref
    
print(A)
print("–" * 20)
A_rref = gaussian_elimination(A)
is_ref(A_rref)

[[-2  4 -2 -1  4]
 [ 4 -8  3 -3  1]
 [ 1 -2  1 -1  1]
 [ 1 -2  0 -3  4]]
––––––––––––––––––––
[[-2.  4. -2. -1.  4.]
 [ 4. -8.  3. -3.  1.]
 [ 1. -2.  1. -1.  1.]
 [ 1. -2.  0. -3.  4.]]
[[ 1.  -2.   1.   0.5 -2. ]
 [ 0.   0.  -1.  -5.   9. ]
 [ 0.   0.   0.  -1.5  3. ]
 [ 0.   0.  -1.  -3.5  6. ]]
––––––––––––––––––––
[[ 1.  -2.   1.   0.5 -2. ]
 [ 0.   0.  -1.  -5.   9. ]
 [ 0.   0.  -1.  -3.5  6. ]
 [ 0.   0.   0.  -1.5  3. ]]
[[ 1.  -2.   0.  -4.5  7. ]
 [-0.  -0.   1.   5.  -9. ]
 [ 0.   0.   0.   1.5 -3. ]
 [ 0.   0.   0.  -1.5  3. ]]
––––––––––––––––––––
[[ 1.  -2.   0.  -4.5  7. ]
 [-0.  -0.   1.   5.  -9. ]
 [ 0.   0.   0.   1.5 -3. ]
 [ 0.   0.   0.  -1.5  3. ]]
[[ 1. -2.  0.  0. -2.]
 [-0. -0.  1.  0.  1.]
 [ 0.  0.  0.  1. -2.]
 [ 0.  0.  0.  0.  0.]]
––––––––––––––––––––
[[ 1. -2.  0.  0. -2.]
 [-0. -0.  1.  0.  1.]
 [ 0.  0.  0.  1. -2.]
 [ 0.  0.  0.  0.  0.]]


np.True_